In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


#  DeBERTa-v3  —  5-Fold CV Multiple-Choice fine-tuning
**AutoTokenizer + cosine LR scheduler + 5-model ensemble + W&B**

In [2]:
!pip install -q -U transformers datasets sentencepiece accelerate

import os, gc, numpy as np, pandas as pd, torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from transformers import (AutoTokenizer, AutoModelForMultipleChoice,
                          TrainingArguments, Trainer)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.5 MB/s eta 0:00:00


# Config


In [3]:
DATA     = "/kaggle/input/competitions/smart-mcq-solver-challenge"
MODEL    = "microsoft/deberta-v3-base"
OPTIONS  = list("ABCDE")
MAXLEN   = 256
N_FOLDS  = 5
EPOCHS   = 3          
LR       = 5e-6       
BATCH    = 4
GRAD_ACC = 4          
SEED     = 42

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Load 

In [4]:
train = pd.read_csv(f"{DATA}/train.csv").reset_index(drop=True)
test  = pd.read_csv(f"{DATA}/test.csv").reset_index(drop=True)
train["label"] = train["answer"].map({c: i for i, c in enumerate(OPTIONS)})
test["label"]  = 0

# Tokenizer + preprocessing


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def preprocess(ex):
    first  = sum([[p] * 5 for p in ex["prompt"]], [])
    second = sum([[ex[o][i] for o in OPTIONS] for i in range(len(ex["prompt"]))], [])
    tok = tokenizer(first, second, truncation=True, max_length=MAXLEN)
    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tok.items()}

drop_cols = ["id", "prompt", "A", "B", "C", "D", "E", "answer"]
full_ds = Dataset.from_pandas(train).map(preprocess, batched=True, remove_columns=drop_cols)

test_drop = [c for c in drop_cols if c in test.columns]
test_ds = Dataset.from_pandas(test).map(preprocess, batched=True, remove_columns=test_drop)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

# Collator

In [6]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    def __call__(self, features):
        name = "label" if "label" in features[0] else "labels"
        labels = [f.pop(name) for f in features]
        n, k = len(features), len(features[0]["input_ids"])
        flat = sum([[{key: f[key][i] for key in f} for i in range(k)] for f in features], [])
        batch = self.tokenizer.pad(flat, padding=self.padding,
                                   max_length=self.max_length, return_tensors="pt")
        batch = {key: v.view(n, k, -1) for key, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

collator = DataCollatorForMultipleChoice(tokenizer)

# Metric

In [7]:
def map3(logits, labels):
    order = np.argsort(-logits, axis=1)
    s = 0.0
    for o, l in zip(order, labels):
        pos = int(np.where(o == l)[0][0])
        if pos < 3:
            s += 1.0 / (pos + 1)
    return s / len(labels)

def compute_metrics(p):
    logits, labels = p
    return {"map@3": map3(logits, labels),
            "acc": float((logits.argmax(1) == labels).mean())}

# W&B

In [8]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="deberta-v3-5fold-cv",
                 config={"model": MODEL, "folds": N_FOLDS, "epochs": EPOCHS,
                         "lr": LR, "eff_batch": BATCH * GRAD_ACC,
                         "scheduler": "cosine", "max_len": MAXLEN})

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ishankgpt02 (ishankgpt02-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260720_110844-356qt69a
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run deberta-v3-5fold-cv
wandb: ⭐️ View project at https://wandb.ai/ishankgpt02-na/23f1002033-t22026
wandb: 🚀 View run at https://wandb.ai/ishankgpt02-na/23f1002033-t22026/runs/356qt69a


# 5-Fold CV


In [9]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_logits  = np.zeros((len(train), 5))
test_logits = np.zeros((len(test), 5))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train["label"])):
    print(f"FOLD {fold+1}/{N_FOLDS}")
    tr_ds = full_ds.select(tr_idx.tolist())
    va_ds = full_ds.select(va_idx.tolist())
    model = AutoModelForMultipleChoice.from_pretrained(MODEL)
    args = TrainingArguments(
        output_dir=f"out_fold{fold}",
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS,
        lr_scheduler_type="cosine",
        warmup_ratio=0.15,                       
        weight_decay=0.01,
        max_grad_norm=0.3,
        fp16=False,
        bf16=False,                               
        eval_strategy="epoch",
        save_strategy="epoch",                    
        load_best_model_at_end=True,              
        metric_for_best_model="map@3",
        greater_is_better=True,
        save_total_limit=1,                       
        report_to="none",
        logging_steps=50,
        seed=SEED,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=tr_ds, eval_dataset=va_ds,
                      processing_class=tokenizer,
                      data_collator=collator,
                      compute_metrics=compute_metrics)
    trainer.train()
    oof_logits[va_idx] = trainer.predict(va_ds).predictions
    fold_map3 = map3(oof_logits[va_idx], train["label"].values[va_idx])
    fold_scores.append(fold_map3)
    print(f"Fold {fold+1} MAP@3: {fold_map3:.4f}")
    wandb.log({"fold": fold + 1, "fold_map@3": fold_map3})

    if fold == 0 and fold_map3 < 0.55:
        print("WARNING: fold 1 near random, lower LR to 5e-6 and re-run.")

    test_logits += trainer.predict(test_ds).predictions / N_FOLDS
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

FOLD 1/5


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight 

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Fold 5 MAP@3: 1.0000


# Overall CV


In [10]:
cv_map3 = map3(oof_logits, train["label"].values)
print(f"OOF CV MAP@3: {cv_map3:.4f}")
print("per-fold:", [round(s, 4) for s in fold_scores])
print("[OK] trained" if cv_map3 > 0.55 else "[FAIL] still random - lower LR")
wandb.log({"cv_map@3": cv_map3, "cv_std": float(np.std(fold_scores))})
run.summary["cv_map@3"] = cv_map3
run.finish()

wandb: updating run metadata


OOF CV MAP@3: 0.9928
per-fold: [0.995, 0.9812, 0.995, 0.9929, 1.0]
[OK] trained


wandb: uploading data
wandb: 
wandb: Run history:
wandb:   cv_map@3 ▁
wandb:     cv_std ▁
wandb:       fold ▁▃▅▆█
wandb: fold_map@3 ▆▁▆▅█
wandb: 
wandb: Run summary:
wandb:   cv_map@3 0.99283
wandb:     cv_std 0.00624
wandb:       fold 5
wandb: fold_map@3 1
wandb: 
wandb: 🚀 View run deberta-v3-5fold-cv at: https://wandb.ai/ishankgpt02-na/23f1002033-t22026/runs/356qt69a
wandb: ⭐️ View project at: https://wandb.ai/ishankgpt02-na/23f1002033-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260720_110844-356qt69a/logs


# Submission (5-fold ensemble)


In [11]:
order = np.argsort(-test_logits, axis=1)
preds = [" ".join(OPTIONS[i] for i in row[:3]) for row in order]
submission = pd.DataFrame({"ID": test["id"], "Prediction": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved (5-fold ensemble)")

submission.csv saved (5-fold ensemble)
